# Grid Search: Batch Size x Inducing Points

Results from `grid_search_batch_sz-inducing_pt.sh` (2026-03-30).

**Sweep:**
- **batch_size**: 16384, 32768, 49152
- **n_inducing**: 256, 512, 1024, 2048, 4096

All runs used 4 GPUs, 50 epochs, 5 MT LOSO sites (same sites across all 3 runs), workers_per_gpu=1.

| Run tag | batch_size | total wall time |
|---------|-----------|----------------|
| 032926_13 | 16,384 | 17.0 hr |
| 033026_1 | 32,768 | 15.5 hr |
| 033026_2 | 49,152 | 15.7 hr |

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

runs_dir = 'runs'

# Grid search runs: batch_size x inducing points
grid_runs = {
    '032926_13': 16384,
    '033026_1': 32768,
    '033026_2': 49152,
}

# Load all data
summaries = []
site_metrics = []
model_params = []
fold_losses = {}

for tag, bs in grid_runs.items():
    run_path = os.path.join(runs_dir, tag)
    
    # Summary (mean across folds, per inducing count)
    s = pd.read_csv(os.path.join(run_path, 'svgp_summary.csv'))
    s['batch_size'] = bs
    s['run_tag'] = tag
    summaries.append(s)
    
    # Per-site metrics
    sm = pd.read_csv(os.path.join(run_path, 'svgp_site_metrics.csv'))
    sm['batch_size'] = bs
    sm['run_tag'] = tag
    site_metrics.append(sm)
    
    # Model params (per fold x inducing)
    mp = pd.read_csv(os.path.join(run_path, 'model_params.csv'))
    mp['batch_size'] = bs
    mp['run_tag'] = tag
    model_params.append(mp)
    
    # Loss curves
    with open(os.path.join(run_path, 'svgp_fold_losses.json')) as f:
        losses = json.load(f)
    fold_losses[bs] = losses

summary_df = pd.concat(summaries, ignore_index=True)
site_df = pd.concat(site_metrics, ignore_index=True)
params_df = pd.concat(model_params, ignore_index=True)

print(f"Loaded {len(grid_runs)} runs")
print(f"Summary: {len(summary_df)} rows (batch_size x n_inducing)")
print(f"Site metrics: {len(site_df)} rows (batch_size x site x n_inducing)")
summary_df[['batch_size', 'n_inducing', 'r2_log', 'r2_orig', 'mean_train_time']].round(3)

## Model Performance: R² by Batch Size x Inducing Points

Mean R² across 5 LOSO folds. Log-scale R² is the primary metric (model trains on log PM2.5).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {16384: '#1f77b4', 32768: '#ff7f0e', 49152: '#2ca02c'}
inducing_vals = sorted(summary_df['n_inducing'].unique())

for metric, ax, title in [('r2_log', axes[0], 'R² (log scale)'), 
                           ('r2_orig', axes[1], 'R² (original scale)')]:
    for bs in sorted(grid_runs.values()):
        sub = summary_df[summary_df['batch_size'] == bs].sort_values('n_inducing')
        ax.plot(sub['n_inducing'], sub[metric], 'o-', label=f'bs={bs:,}', 
                color=colors[bs], linewidth=2, markersize=7)
    ax.set_xlabel('Inducing points (M)')
    ax.set_ylabel(metric.replace('_', ' '))
    ax.set_title(title)
    ax.set_xscale('log', base=2)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grid_search_r2_by_bs_inducing.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the best configs
print("Top 5 configs by mean R² (log scale):")
top = summary_df.nlargest(5, 'r2_log')[['batch_size', 'n_inducing', 'r2_log', 'r2_orig', 'rmse_log']]
print(top.to_string(index=False, float_format='%.4f'))

## Per-Site Performance Breakdown

Some sites are much harder than others. Look at R² by site to see whether the batch_size / inducing point trends are consistent or driven by specific sites.

In [ ]:
sites = site_df['site'].unique()
n_sites = len(sites)

fig, axes = plt.subplots(n_sites, 2, figsize=(14, 4 * n_sites), sharex='col')

for i, site in enumerate(sites):
    for j, (metric, title) in enumerate([('r2_log', 'R² (log)'), ('r2_orig', 'R² (orig)')]):
        ax = axes[i, j]
        for bs in sorted(grid_runs.values()):
            sub = site_df[(site_df['site'] == site) & (site_df['batch_size'] == bs)].sort_values('n_inducing')
            ax.plot(sub['n_inducing'], sub[metric], 'o-', label=f'bs={bs:,}', 
                    color=colors[bs], linewidth=1.5, markersize=5)
        ax.set_title(f'{site} ({site_df[site_df["site"]==site]["state"].iloc[0]}) — {title}')
        ax.set_xscale('log', base=2)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        if i == 0:
            ax.legend(fontsize=8)
        if i == n_sites - 1:
            ax.set_xlabel('Inducing points (M)')

plt.tight_layout()
plt.savefig('grid_search_r2_per_site.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: mean R² (log) across batch_size x n_inducing
r2_pivot = summary_df.pivot(index='batch_size', columns='n_inducing', values='r2_log')

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(r2_pivot.values, cmap='RdYlGn', aspect='auto', vmin=0.35, vmax=0.55)

ax.set_xticks(range(len(r2_pivot.columns)))
ax.set_xticklabels([f'{m:,}' for m in r2_pivot.columns])
ax.set_yticks(range(len(r2_pivot.index)))
ax.set_yticklabels([f'{bs:,}' for bs in r2_pivot.index])
ax.set_xlabel('Inducing Points (M)')
ax.set_ylabel('Batch Size')
ax.set_title('Mean R² (log scale) — 5-fold LOSO')

for i in range(len(r2_pivot.index)):
    for j in range(len(r2_pivot.columns)):
        val = r2_pivot.iloc[i, j]
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontweight='bold', fontsize=11)

plt.colorbar(im, ax=ax, label='R² (log)')
plt.tight_layout()
plt.savefig('grid_search_r2_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Training Time

How training time scales with inducing points and batch size. Time is mean per-fold training time (seconds) across 5 folds, each running 50 epochs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Training time vs inducing points
ax = axes[0]
for bs in sorted(grid_runs.values()):
    sub = summary_df[summary_df['batch_size'] == bs].sort_values('n_inducing')
    ax.plot(sub['n_inducing'], sub['mean_train_time'] / 60, 'o-', label=f'bs={bs:,}', 
            color=colors[bs], linewidth=2, markersize=7)
ax.set_xlabel('Inducing points (M)')
ax.set_ylabel('Mean training time per fold (min)')
ax.set_title('Training Time vs Inducing Points')
ax.set_xscale('log', base=2)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Total wall time (including IP selection)
ax = axes[1]
for bs in sorted(grid_runs.values()):
    sub = summary_df[summary_df['batch_size'] == bs].sort_values('n_inducing')
    ax.plot(sub['n_inducing'], sub['mean_fold_time'] / 60, 'o-', label=f'bs={bs:,}', 
            color=colors[bs], linewidth=2, markersize=7)
ax.set_xlabel('Inducing points (M)')
ax.set_ylabel('Mean total fold time (min)')
ax.set_title('Total Fold Time (train + IP selection)')
ax.set_xscale('log', base=2)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: R² vs training time (efficiency frontier)
ax = axes[2]
for bs in sorted(grid_runs.values()):
    sub = summary_df[summary_df['batch_size'] == bs].sort_values('n_inducing')
    ax.plot(sub['mean_train_time'] / 60, sub['r2_log'], 'o-', label=f'bs={bs:,}', 
            color=colors[bs], linewidth=2, markersize=7)
    # Annotate M values
    for _, row in sub.iterrows():
        ax.annotate(f'M={int(row["n_inducing"])}', 
                    (row['mean_train_time'] / 60, row['r2_log']),
                    textcoords="offset points", xytext=(5, 5), fontsize=7)
ax.set_xlabel('Mean training time per fold (min)')
ax.set_ylabel('R² (log scale)')
ax.set_title('Performance vs Cost')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grid_search_time_vs_performance.png', dpi=150, bbox_inches='tight')
plt.show()

# Print timing table
print("Mean training time per fold (minutes):")
time_pivot = summary_df.pivot(index='batch_size', columns='n_inducing', values='mean_train_time') / 60
print(time_pivot.round(1).to_string())
print()
print("Total wall time per run (hours):")
for bs in sorted(grid_runs.values()):
    sub = summary_df[summary_df['batch_size'] == bs]
    total = sub['total_wall_time'].sum()
    print(f"  bs={bs:>6,}: {total/3600:.1f} hr")

## Loss Curves

Check convergence: are 50 epochs enough? Did any configs need more training?

In [ ]:
# Plot loss curves for fold 0 (site 108720_47582) across all batch sizes and inducing counts
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
inducing_colors = {256: '#1f77b4', 512: '#ff7f0e', 1024: '#2ca02c', 2048: '#d62728', 4096: '#9467bd'}

for ax, bs in zip(axes, sorted(grid_runs.values())):
    losses = fold_losses[bs]
    for M in [256, 512, 1024, 2048, 4096]:
        key = f'108720_47582_M{M}'
        if key in losses:
            ax.plot(losses[key], label=f'M={M:,}', color=inducing_colors[M], linewidth=1.5, alpha=0.8)
    ax.set_title(f'batch_size={bs:,}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ELBO Loss' if ax == axes[0] else '')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Loss Curves — Fold 0 (site 108720_47582)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('grid_search_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## GPU Memory & Scaling Recommendations

From the earlier batch_size x workers_per_gpu grid search (032926_1 through 032926_12, 1-epoch timing runs on 48 GiB GPUs):

| batch_size | workers_per_gpu=1 | wpg=2 | wpg=3 | wpg=4 |
|-----------|------------------|-------|-------|-------|
| 16,384 | 16.3 GiB | 31.4 GiB | 46.5 GiB | OOM |
| 32,768 | 27.8 GiB | OOM | OOM | OOM |
| 49,152 | 40.1 GiB | OOM | OOM | OOM |

**For 20 GiB GPUs** (40 available): only `batch_size=16384, workers_per_gpu=1` fits safely.
**For 48 GiB GPUs** (8 available): all three batch sizes work with `workers_per_gpu=1`. `bs=16384` could also fit `wpg=2`.

In [ ]:
# Production run time estimates
# Scenario: full LOSO CV over all MT sites (~56 sites) or full CONUS (~932 sites)

print("=" * 80)
print("PRODUCTION RUN TIME ESTIMATES")
print("=" * 80)

# Per-fold time for M=256 (best R²) from each batch_size
print("\nPer-fold training time at M=256 (best-performing inducing count):")
for bs in sorted(grid_runs.values()):
    sub = site_df[(site_df['batch_size'] == bs) & (site_df['n_inducing'] == 256)]
    mean_train = sub['train_time'].mean()
    mean_total = sub['total_time'].mean()
    print(f"  bs={bs:>6,}: train={mean_train/60:.1f} min, total (w/ IP selection)={mean_total/60:.1f} min")

print("\n" + "-" * 80)
print("Scenario: 186 MT/ID/WY sites, M=256, 50 epochs, workers_per_gpu=1")
print("-" * 80)

N_FOLDS = 186
configs = [
    ("20 GiB GPUs (40x)", 16384, 40),
    ("48 GiB GPUs (8x)", 32768, 8),
    ("48 GiB GPUs (8x)", 49152, 8),
    ("Mixed: 40x 20GiB + 8x 48GiB", 16384, 48),  # all GPUs, bs=16384
]

for label, bs, n_gpus in configs:
    sub = site_df[(site_df['batch_size'] == bs) & (site_df['n_inducing'] == 256)]
    mean_fold_time = sub['total_time'].mean()
    
    folds_per_gpu = np.ceil(N_FOLDS / n_gpus)
    wall_time_s = folds_per_gpu * mean_fold_time
    wall_time_hr = wall_time_s / 3600
    
    print(f"\n  {label}, bs={bs:,}:")
    print(f"    {folds_per_gpu:.0f} folds/GPU x {mean_fold_time/60:.1f} min/fold = {wall_time_hr:.1f} hours")

print("\n" + "-" * 80)
print("Scenario: 932 CONUS sites, M=256, 50 epochs, workers_per_gpu=1")
print("-" * 80)

N_FOLDS = 932
for label, bs, n_gpus in configs:
    sub = site_df[(site_df['batch_size'] == bs) & (site_df['n_inducing'] == 256)]
    mean_fold_time = sub['total_time'].mean()
    
    folds_per_gpu = np.ceil(N_FOLDS / n_gpus)
    wall_time_s = folds_per_gpu * mean_fold_time
    wall_time_hr = wall_time_s / 3600
    wall_time_days = wall_time_hr / 24
    
    print(f"\n  {label}, bs={bs:,}:")
    print(f"    {folds_per_gpu:.0f} folds/GPU x {mean_fold_time/60:.1f} min/fold = {wall_time_hr:.1f} hr ({wall_time_days:.1f} days)")

## Learned Kernel Parameters

Do the learned hyperparameters vary systematically with batch size or inducing points?

In [ ]:
param_cols = ['base_scale', 'summer_scale', 'winter_scale', 'seasonal_scale', 'noise']

fig, axes = plt.subplots(1, len(param_cols), figsize=(18, 4))

for ax, param in zip(axes, param_cols):
    for bs in sorted(grid_runs.values()):
        sub = params_df[params_df['batch_size'] == bs].groupby('n_inducing')[param].mean()
        ax.plot(sub.index, sub.values, 'o-', label=f'bs={bs:,}', color=colors[bs], linewidth=1.5)
    ax.set_title(param)
    ax.set_xscale('log', base=2)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.grid(True, alpha=0.3)
    if ax == axes[0]:
        ax.legend(fontsize=8)

plt.suptitle('Mean Learned Kernel Parameters (across 5 folds)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('grid_search_kernel_params.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary & Recommendations

### Performance
- **Best R² (log): M=256** consistently across all batch sizes. Performance *degrades* with more inducing points, especially at sites 105103_48308 and 108512_45780 where R² goes negative at M>=1024. This suggests overfitting with too many inducing points on these 5 MT sites.
- **Batch size has minimal effect on R²**: all three batch sizes give similar performance at each M. The differences are within noise (bs=32768 has a slight edge at M=256 with R²=0.516 vs 0.496/0.490).

### Timing
- Training time is dominated by inducing point count: M=4096 is ~13x slower than M=256.
- Batch size effect on speed is modest: bs=49152 is ~10-15% faster per fold than bs=16384 (fewer batches per epoch), but only the 48 GiB GPUs can run it.
- All models ran the full 50 epochs (no early stopping triggered) — consider increasing `n_epochs` or tuning `patience` to check if models are still improving.

### Recommended configs for production LOSO

| GPU type | batch_size | n_inducing | workers_per_gpu | notes |
|---------|-----------|------------|----------------|-------|
| 20 GiB (40 GPUs) | 16,384 | 256 | 1 | Only viable batch size; ~16 GiB peak memory |
| 48 GiB (8 GPUs) | 32,768 | 256 | 1 | Slightly faster per fold; ~28 GiB peak |

**Simplest approach**: use `batch_size=16384, M=256, workers_per_gpu=1` across all 48 GPUs. This maximizes parallelism (48 folds at once) and uses the best-performing inducing count.